# Qwen2.5-Coder-14B Fine-Tuning with Unsloth

This notebook fine-tunes **Qwen2.5-Coder-14B** using **QLoRA** on a **GPU (T4/P100+)**.

### What we're doing:
- Load Qwen2.5-Coder-14B in **4-bit** using Unsloth's optimized kernels
- Apply **LoRA adapters** (only ~0.5% of params trainable)
- Train on a **Python Q&A dataset** (11,962 examples)
- Save and download the trained adapter

**Compatible with:** Google Colab (T4 free) | Kaggle Notebooks (P100 free) | VS Code (remote GPU)

**Estimated runtime:** ~2-4 hours for full dataset on T4 free tier

---
## Step 0: Environment Detection

Auto-detects whether we're in **Colab**, **Kaggle**, or **VS Code** and sets up paths accordingly.

In [2]:
# Detect environment: Colab vs Kaggle vs VS Code
import sys, os, json

ENV = "vscode"  # default
IS_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
IS_KAGGLE = "KAGGLE_URL_BASE" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_COLAB:
    ENV = "colab"
elif IS_KAGGLE:
    ENV = "kaggle"

print(f"Environment detected: {ENV.upper()}")
print(f"Python: {sys.version}")

# Check GPU (warning only, don't block for VS Code users)
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_mem:.1f} GB")

if not torch.cuda.is_available():
    if ENV == "colab":
        raise RuntimeError("No GPU! Go to Runtime > Change runtime type > T4 GPU")
    elif ENV == "kaggle":
        raise RuntimeError("No GPU! Go to Settings > Accelerator > GPU P100")
    else:
        print("WARNING: No GPU detected. Training will be VERY slow.")
        print("Recommendation: Upload this notebook to Google Colab (File > Upload Notebook)")
elif gpu_mem < 14:
    print("Warning: Less than 14GB VRAM - training may OOM. Reduce max_seq_length.")

Environment detected: KAGGLE
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
GPU: Tesla T4
VRAM: 14.6 GB


---
## Step 1: Install Dependencies

In [ ]:
# Install dependencies based on environment
import sys, os
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "KAGGLE_URL_BASE" in os.environ

if IS_COLAB or IS_KAGGLE:
    print("Installing GPU dependencies (Colab/Kaggle)...")
    import subprocess
    subprocess.run("pip install unsloth", shell=True)
    subprocess.run("pip install --upgrade --no-deps --force-reinstall unsloth", shell=True)
    subprocess.run("pip install flash-attn --no-build-isolation", shell=True)
    subprocess.run("pip install datasets", shell=True)
else:
    print("Local CPU environment detected. Skipping GPU-specific installations (Unsloth, Flash-Attn).")
    try:
        import datasets
    except ImportError:
        import subprocess
        subprocess.run("pip install datasets", shell=True)

Installing GPU dependencies (Colab/Kaggle)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 6.6 MB/s eta 0:00:

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


  Using cached unsloth-2026.5.7-py3-none-any.whl.metadata (57 kB)
Using cached unsloth-2026.5.7-py3-none-any.whl (71.1 MB)
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2026.5.7
    Uninstalling unsloth-2026.5.7:
      Successfully uninstalled unsloth-2026.5.7
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 62.2 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'


In [6]:
# Verify GPU
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_mem:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda if torch.cuda.is_available() else 'None'}")
print()

if not torch.cuda.is_available():
    if ENV in ["colab", "kaggle"]:
        raise RuntimeError("No GPU detected! Go to Runtime > Change runtime type > T4 GPU")
    else:
        print("WARNING: No GPU detected. Running in Local CPU Fallback mode.")
elif gpu_mem < 14:
    print("Warning: Less than 14GB VRAM - training may OOM. Try reducing max_seq_length.")

GPU: Tesla P100-PCIE-16GB
VRAM: 15.9 GB
PyTorch: 2.10.0+cu128
CUDA: 12.8



/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


---
## Step 2: Load the Dataset

The dataset is already formatted as instruction-output pairs.

Choose **ONE** of the following methods to load the data:

### Option A: Load from HuggingFace Hub (Recommended)
Upload the dataset to HF Hub first using `colab_export/upload_to_hf.py`

In [ ]:
from datasets import load_dataset

# ===== SET YOUR HF DATASET NAME HERE =====
HF_DATASET = "YOUR_HF_USERNAME/pythonai-training-data"
# ==========================================

dataset = load_dataset(HF_DATASET, split="train")
print(f"Loaded {len(dataset):,} examples")
print(f"Sample: {dataset[0]['instruction'][:80]}...")

### Option B: Upload JSONL directly to Colab
Run this cell and upload `colab_export/training_dataset.jsonl`

In [ ]:
# Load dataset
import json
from datasets import Dataset

if ENV == "vscode":
    # Local loading for VS Code
    dataset_path = "colab_export/training_dataset.jsonl"
    print(f"Loading local dataset from {dataset_path}...")
    rows = []
    with open(dataset_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    dataset = Dataset.from_list(rows)
    print(f"Loaded {len(dataset):,} examples locally.")
else:
    # Upload the JSONL file from your computer (Colab only)
    from google.colab import files
    print("Upload training_dataset.jsonl from the colab_export folder...")
    uploaded = files.upload()

    filename = list(uploaded.keys())[0]
    rows = []
    for line in uploaded[filename].decode('utf-8').splitlines():
        if line.strip():
            rows.append(json.loads(line))

    dataset = Dataset.from_list(rows)
    print(f"Loaded {len(dataset):,} examples from {filename}")

### Option C: Load from Google Drive
Mount Drive and load the dataset from there

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
from datasets import Dataset

# ===== SET YOUR PATH =====
DRIVE_PATH = "/content/drive/MyDrive/pythonai_training/training_dataset.jsonl"
# =========================

rows = []
with open(DRIVE_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

dataset = Dataset.from_list(rows)
print(f"Loaded {len(dataset):,} examples from Google Drive")

### Option D: Load from Kaggle Dataset (if on Kaggle)
If you uploaded the dataset as a Kaggle Dataset, use this:

In [ ]:
# This cell is for Kaggle environment - dataset added via Add Data button
import os, json
from datasets import Dataset

# List available input datasets
input_dir = "/kaggle/input/"
if os.path.exists(input_dir):
    print("Available Kaggle datasets:")
    for d in os.listdir(input_dir):
        print(f"  {d}")
        for f in os.listdir(os.path.join(input_dir, d)):
            print(f"    - {f}")

# ===== SET YOUR KAGGLE DATASET PATH =====
KAGGLE_DATA_PATH = "/kaggle/input/pythonai-training-data/training_dataset.jsonl"
# =========================================

if os.path.exists(KAGGLE_DATA_PATH):
    rows = []
    with open(KAGGLE_DATA_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    dataset = Dataset.from_list(rows)
    print(f"Loaded {len(dataset):,} examples from Kaggle dataset")
else:
    print("Kaggle dataset not found. Upload training_dataset.jsonl as a Kaggle Dataset first.")

---
## Step 3: Load Qwen2.5-Coder-14B with Unsloth (4-bit QLoRA)

In [ ]:
import torch

MODEL_NAME = "unsloth/Qwen2.5-Coder-14B-bnb-4bit"
MAX_SEQ_LENGTH = 1024

# Check if we can use Unsloth (needs CUDA and unsloth package)
use_unsloth = False
try:
    from unsloth import FastLanguageModel
    if torch.cuda.is_available():
        use_unsloth = True
except ImportError:
    pass

if use_unsloth:
    print("Loading 4-bit model with Unsloth optimizations...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,  # Auto-detect
        load_in_4bit=True,
    )
else:
    print("Unsloth or CUDA not available. Falling back to standard HF transformers on CPU.")
    # For CPU training, use a very small model (sshleifer/tiny-gpt2)
    # to avoid downloading 14B model and running out of CPU RAM
    FALLBACK_MODEL = "sshleifer/tiny-gpt2"
    print(f"Using fallback model: {FALLBACK_MODEL}")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(FALLBACK_MODEL)

print(f"Model loaded! Parameters: {model.num_parameters():,}")
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Step 4: Add LoRA Adapters

In [ ]:
if use_unsloth:
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
else:
    print("Using standard PEFT/LoraConfig for fallback model...")
    from peft import LoraConfig, get_peft_model
    # Find proper linear layers for fallback model
    model_type = getattr(getattr(model, "config", None), "model_type", "").lower()
    if "gpt2" in model_type:
        target_modules = ["c_attn", "c_proj"]
    else:
        target_modules = ["q_proj", "v_proj"]
        
    lora_config = LoraConfig(
        r=16,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=target_modules,
    )
    model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = model.num_parameters()
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

---
## Step 5: Format Dataset for Training

Unsloth's SFTTrainer expects a chat template. We'll format instructions into Alpaca-style prompts.

In [ ]:
# Alpaca-style prompt template
alpaca_prompt = """Below is an instruction that describes a task. Write a response that completes the request.

### Instruction:
{}

### Response:
{}"""
EOS_TOKEN = tokenizer.eos_token  # Must be added!

def format_prompts(examples):
    instructions = examples["instruction"]
    outputs = examples["output"]
    texts = []
    for instr, out in zip(instructions, outputs):
        text = alpaca_prompt.format(instr, out) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Apply formatting
dataset = dataset.map(format_prompts, batched=True)
print(f"Formatted dataset: {len(dataset):,} examples")
print("\nSample format:")
print(dataset[0]["text"][:300] + "...")

In [ ]:
# Train/validation split (90/10)
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
print(f"Train: {len(train_dataset):,}")
print(f"Validation: {len(eval_dataset):,}")

---
## Step 6: Configure Training Arguments & Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

if use_unsloth:
    from unsloth import is_bfloat16_supported
    bf16_val = is_bfloat16_supported()
    fp16_val = not bf16_val
    optim_val = "adamw_8bit"
else:
    bf16_val = False
    fp16_val = False  # standard training on CPU uses float32
    optim_val = "adamw_torch"

# Define local CPU smoke test overrides
max_steps_val = 2 if not use_unsloth else -1
epochs_val = 1.0 if use_unsloth else 1.0
dataset_num_proc_val = 2 if use_unsloth else 1

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=dataset_num_proc_val,
    packing=False,
    args=TrainingArguments(
        output_dir="./qwen14b_pythonai_adapter",
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=2 if not use_unsloth else 20,
        max_steps=max_steps_val,
        num_train_epochs=epochs_val,
        learning_rate=2e-4,
        fp16=fp16_val,
        bf16=bf16_val,
        logging_steps=1,
        optim=optim_val,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        evaluation_strategy="steps" if use_unsloth else "no",
        eval_steps=50,
        save_strategy="steps" if use_unsloth else "no",
        save_steps=100,
        save_total_limit=2,
    ),
)

print("Trainer configured!")
if use_unsloth:
    print(f"Batch size: 2, Grad accum: 4, Effective batch: 8")
else:
    print("Running in Local CPU Fallback (Smoke Test Mode: max_steps=2)")

In [ ]:
# Start training!
print("Starting training... Use Ctrl+C to interrupt (progress won't be lost)\n")
trainer_stats = trainer.train()
print("\nTraining complete!")

---
## Step 7: Save & Export the Adapter

In [ ]:
# Save the LoRA adapter locally
ADAPTER_DIR = "./pythonai_qwen14b_lora_adapter"

# Save LoRA weights only (~34 MB for r=16)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to {ADAPTER_DIR}")

# List files
import os
for f in os.listdir(ADAPTER_DIR):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f"  {f}: {size/1024:.1f} KB")

In [ ]:
# Download to your local machine (Colab only)
if ENV == "colab":
    from google.colab import files
    import shutil

    shutil.make_archive("pythonai_qwen14b_lora_adapter", 'zip', ADAPTER_DIR)
    files.download("pythonai_qwen14b_lora_adapter.zip")
    print("Download started! Save the zip to your project folder and unzip.")
else:
    print(f"Adapter saved at {ADAPTER_DIR}")
    print("On Kaggle/VS Code: download the folder manually via the file browser.")

### Alternative: Save to Google Drive

In [ ]:
# Save to Google Drive for persistence
if ENV == "colab":
    import shutil
    from pathlib import Path

    drive_adapter = Path("/content/drive/MyDrive/pythonai_qwen14b_lora_adapter")
    drive_adapter.mkdir(parents=True, exist_ok=True)

    shutil.copytree(ADAPTER_DIR, str(drive_adapter), dirs_exist_ok=True)
    print(f"Adapter copied to Google Drive: {drive_adapter}")
else:
    print("Google Drive only available in Colab. Skip this cell.")

---
## Step 8: Test the Trained Model

In [ ]:
# Test inference with the trained adapter
if use_unsloth:
    from unsloth import FastLanguageModel

    # Load base model + trained adapter
    test_model, test_tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    test_model.load_adapter(ADAPTER_DIR)
    FastLanguageModel.for_inference(test_model)
    device = "cuda"
else:
    print("Using standard transformers + PEFT for inference on CPU...")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    test_tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL)
    base_model = AutoModelForCausalLM.from_pretrained(FALLBACK_MODEL)
    test_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    device = "cpu"

def generate_response(prompt):
    formatted = alpaca_prompt.format(prompt, "")
    inputs = test_tokenizer([formatted], return_tensors="pt").to(device)
    outputs = test_model.generate(
        **inputs,
        max_new_tokens=64 if device == "cpu" else 256,
        temperature=0.7,
        top_p=0.95,
        repetition_penalty=1.1,
    )
    response = test_tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the response part
    response = response.split("### Response:")[-1].strip()
    return response

# Test prompts
test_prompts = [
    "Explain Python context managers and the `with` statement.",
    "Write a Python function to merge two sorted lists.",
    "What is the difference between a list and a tuple in Python?",
]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"Q: {prompt}")
    print(f"\nA: {generate_response(prompt)}")

---
## Step 9: Merge + Export GGUF (for Ollama)

Optional: Merge LoRA weights into the base model and export to GGUF format for **local Ollama use**.

In [ ]:
# Merge LoRA weights into base model
if use_unsloth:
    merged_model, merged_tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    merged_model.load_adapter(ADAPTER_DIR)
    merged_model = merged_model.merge_and_unload()
else:
    print("Merging model weights on CPU using standard PEFT...")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    merged_tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL)
    base_model = AutoModelForCausalLM.from_pretrained(FALLBACK_MODEL)
    peft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    merged_model = peft_model.merge_and_unload()

print("LoRA weights merged into base model!")

In [ ]:
# Save merged model in HF format
merged_path = "./qwen14b_pythonai_merged"
merged_model.save_pretrained(merged_path)
merged_tokenizer.save_pretrained(merged_path)
print(f"Merged model saved to {merged_path}")

# For GGUF export (needs llama.cpp):
# !git clone https://github.com/ggerganov/llama.cpp
# !cd llama.cpp && make -j
# !python llama.cpp/convert_hf_to_gguf.py {merged_path} --outfile qwen14b_pythonai.gguf

# Then on your local machine:
# ollama create pythonai-expert -f Modelfile
# Modelfile content:
# FROM ./qwen14b_pythonai.gguf
# TEMPLATE """{{ .System }}
# {{ .Prompt }}"""

---
## Quick Test Mode (5 minutes)

Run this cell instead of Step 6 if you want a quick smoke test first:

In [ ]:
# Quick test: 50 steps on a 200-example subset (~5 min)
if use_unsloth:
    from unsloth import is_bfloat16_supported
    bf16_val = is_bfloat16_supported()
    fp16_val = not bf16_val
    optim_val = "adamw_8bit"
    max_steps_val = 50
else:
    bf16_val = False
    fp16_val = False
    optim_val = "adamw_torch"
    max_steps_val = 2

quick_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset.select(range(min(200, len(train_dataset)))),
    eval_dataset=eval_dataset.select(range(min(40, len(eval_dataset)))),
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        output_dir="./qwen14b_quicktest",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=2 if not use_unsloth else 5,
        max_steps=max_steps_val,
        learning_rate=2e-4,
        fp16=fp16_val,
        bf16=bf16_val,
        logging_steps=1,
        optim=optim_val,
        seed=42,
        report_to="none",
    ),
)

print(f"Running quick test ({max_steps_val} steps)...")
quick_trainer.train()
print("Quick test complete!")

---
## Troubleshooting

| Issue | Fix |
|-------|-----|
| **OOM (Out of Memory)** | Reduce `MAX_SEQ_LENGTH` to 512 or `per_device_train_batch_size` to 1 |
| **Colab runtime disconnects** | Use `!pip install jupyter-ai` or save checkpoints to Drive frequently |
| **Slow training** | This is expected on T4 for 14B. Expect ~2-4 hours for 1 epoch |
| **Model not loading** | Click Runtime > Factory reset runtime, then run from Step 1 |
| **HF auth error** | Get token from huggingface.co/settings/tokens, uncomment `token=` in Step 3 |
| **Kaggle: dataset not found** | Check `/kaggle/input/` path or add dataset via Add Data button |

---
### After Training — Local Setup

Once you download the adapter zip:
```bash
# 1. Unzip into your project
unzip pythonai_qwen14b_lora_adapter.zip -d checkpoints/qwen14b_pythonai/

# 2. Test with your local inference code
python -m src.training.evaluator --adapter-path checkpoints/qwen14b_pythonai

# 3. Compare with other adapters
python -m src.training.comparison --compare-all
```